# Polynomial Regression — Auto MPG Dataset

**Objective:** Predict Miles Per Gallon (MPG) based on engine displacement using polynomial regression of varying degrees, and compare with simple linear regression.

**Methods:**
1. Linear Regression (Degree 1 — baseline)
2. Polynomial Regression (Degrees 2, 3, 4, 5)

**Evaluation Metrics:** MSE, RMSE, R² Score

---
## Cell 1 — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("All libraries imported successfully!")

---
## Cell 2 — Load the Auto MPG Dataset

In [ ]:
# Load Auto MPG dataset directly from UCI repository
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"

column_names = [
    'mpg', 'cylinders', 'displacement', 'horsepower', 'weight',
    'acceleration', 'model_year', 'origin', 'car_name'
]

df = pd.read_csv(
    url,
    delim_whitespace=True,
    names=column_names,
    na_values='?'  # '?' marks missing values in this dataset
)

print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

---
## Cell 3 — Explore the Dataset

In [ ]:
print("=" * 60)
print("DATASET EXPLORATION")
print("=" * 60)

print(f"\nTotal samples: {len(df)}")
print(f"\nMissing values per column:")
print(df.isnull().sum())

print(f"\nData types:")
print(df.dtypes)

print("\n--- Feature: displacement (engine size in cubic inches) ---")
print(f"Mean:   {df['displacement'].mean():.2f}")
print(f"Median: {df['displacement'].median():.2f}")
print(f"Std:    {df['displacement'].std():.2f}")
print(f"Range:  {df['displacement'].min():.1f} — {df['displacement'].max():.1f}")

print("\n--- Target: mpg (miles per gallon) ---")
print(f"Mean:   {df['mpg'].mean():.2f}")
print(f"Median: {df['mpg'].median():.2f}")
print(f"Range:  {df['mpg'].min():.1f} — {df['mpg'].max():.1f}")

---
## Cell 4 — Data Preprocessing

In [ ]:
print("=" * 60)
print("DATA PREPROCESSING")
print("=" * 60)

# ---- Step 1: Handle missing values ----
print(f"\nStep 1: Handle missing values")
print(f"  Missing before: {df.isnull().sum().sum()} (all in 'horsepower')")
df = df.dropna()  # Only 6 rows have missing horsepower — safe to drop
print(f"  Rows after dropping: {len(df)}")

# ---- Step 2: Select feature and target ----
print(f"\nStep 2: Select feature and target")
X = df[['displacement']].values  # Shape: (n, 1)
y = df['mpg'].values             # Shape: (n,)
print(f"  Feature: displacement (engine size)")
print(f"  Target:  mpg")
print(f"  Samples: {X.shape[0]}")

# ---- Step 3: Train-Test Split (80/20) ----
print(f"\nStep 3: Train-Test Split (80/20)")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"  Training set: {X_train.shape[0]} samples")
print(f"  Testing set:  {X_test.shape[0]} samples")

print("\nPreprocessing complete!")

---
## Cell 5 — Visualize the Raw Data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot: Displacement vs MPG
axes[0].scatter(X, y, alpha=0.5, s=20, color='steelblue', edgecolors='white', linewidth=0.5)
axes[0].set_xlabel('Engine Displacement (cubic inches)')
axes[0].set_ylabel('Miles Per Gallon (MPG)')
axes[0].set_title('Displacement vs MPG — Raw Data')

# Distribution of MPG
axes[1].hist(y, bins=30, color='steelblue', edgecolor='white', alpha=0.7)
axes[1].set_xlabel('Miles Per Gallon (MPG)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of MPG')

plt.tight_layout()
plt.show()

print("Key Observation: The relationship between displacement and MPG is clearly")
print("NON-LINEAR (curved). A straight line won't fit this well — polynomial will!")
print("\nHuman Analogy: Bigger engine ≠ proportionally less fuel efficiency.")
print("The drop is steep for small engines, then flattens for large engines (diminishing effect).")

---
## Cell 6 — Implement Linear Regression (Degree 1 — Baseline)

In [ ]:
# ---- Linear Regression (baseline) ----
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

# Predictions
y_train_pred_linear = linear_model.predict(X_train)
y_test_pred_linear = linear_model.predict(X_test)

# Metrics
train_mse_linear = mean_squared_error(y_train, y_train_pred_linear)
test_mse_linear = mean_squared_error(y_test, y_test_pred_linear)
train_r2_linear = r2_score(y_train, y_train_pred_linear)
test_r2_linear = r2_score(y_test, y_test_pred_linear)

print("=" * 50)
print("LINEAR REGRESSION (Degree 1 — Baseline)")
print("=" * 50)
print(f"Equation: MPG = {linear_model.coef_[0]:.4f} × displacement + {linear_model.intercept_:.4f}")
print(f"\n{'Metric':<15} {'Train':>10} {'Test':>10}")
print("-" * 37)
print(f"{'MSE':<15} {train_mse_linear:>10.4f} {test_mse_linear:>10.4f}")
print(f"{'RMSE':<15} {np.sqrt(train_mse_linear):>10.4f} {np.sqrt(test_mse_linear):>10.4f}")
print(f"{'R²':<15} {train_r2_linear:>10.4f} {test_r2_linear:>10.4f}")

---
## Cell 7 — Implement Polynomial Regression (Degrees 2, 3, 4, 5)

In [ ]:
# ---- Polynomial Regression for multiple degrees ----

degrees = [2, 3, 4, 5]
poly_models = {}   # Store models
results = []       # Store metrics

# Add linear regression result first
results.append({
    'Degree': 1,
    'Train MSE': train_mse_linear,
    'Test MSE': test_mse_linear,
    'Train RMSE': np.sqrt(train_mse_linear),
    'Test RMSE': np.sqrt(test_mse_linear),
    'Train R²': train_r2_linear,
    'Test R²': test_r2_linear
})

print("=" * 70)
print("POLYNOMIAL REGRESSION — TRAINING MODELS")
print("=" * 70)

for degree in degrees:
    # Create a pipeline: PolynomialFeatures → StandardScaler → LinearRegression
    poly_pipeline = Pipeline([
        ('poly_features', PolynomialFeatures(degree=degree, include_bias=False)),
        ('scaler', StandardScaler()),
        ('linear_reg', LinearRegression())
    ])
    
    # Fit the model
    poly_pipeline.fit(X_train, y_train)
    
    # Predictions
    y_train_pred = poly_pipeline.predict(X_train)
    y_test_pred = poly_pipeline.predict(X_test)
    
    # Metrics
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    # Store
    poly_models[degree] = poly_pipeline
    results.append({
        'Degree': degree,
        'Train MSE': train_mse,
        'Test MSE': test_mse,
        'Train RMSE': np.sqrt(train_mse),
        'Test RMSE': np.sqrt(test_mse),
        'Train R²': train_r2,
        'Test R²': test_r2
    })
    
    print(f"\nDegree {degree}: Train R² = {train_r2:.4f}, Test R² = {test_r2:.4f}, Test MSE = {test_mse:.4f}")

print("\nAll models trained successfully!")

---
## Cell 8 — Comparison Table: Linear vs Polynomial

In [ ]:
results_df = pd.DataFrame(results)

print("=" * 80)
print("COMPARISON: LINEAR vs POLYNOMIAL REGRESSION")
print("=" * 80)
print()
print(results_df.to_string(index=False, float_format='%.4f'))

# Highlight best model
best_idx = results_df['Test R²'].idxmax()
best = results_df.loc[best_idx]
print(f"\n{'='*80}")
print(f"BEST MODEL: Degree {int(best['Degree'])}")
print(f"  Test R²:   {best['Test R²']:.4f}")
print(f"  Test MSE:  {best['Test MSE']:.4f}")
print(f"  Test RMSE: {best['Test RMSE']:.4f}")
print(f"{'='*80}")

print("\nHuman Analogy:")
print("  Degree 1 = Drawing a straight line through curved data (too simple)")
print("  Degree 2 = Capturing the curve (just right for this data)")
print("  Degree 5 = Memorizing every wiggle (risks overfitting — like a student")
print("             who memorizes answers instead of understanding concepts)")

---
## Cell 9 — Visualization: All Polynomial Fits on One Plot

In [ ]:
# Generate smooth x values for plotting curves
X_plot = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)

colors = {
    1: '#E74C3C',   # Red — Linear
    2: '#2ECC71',   # Green — Degree 2
    3: '#3498DB',   # Blue — Degree 3
    4: '#F39C12',   # Orange — Degree 4
    5: '#9B59B6',   # Purple — Degree 5
}

plt.figure(figsize=(12, 7))

# Scatter plot of all data
plt.scatter(X_test, y_test, alpha=0.5, s=25, color='gray',
            edgecolors='white', linewidth=0.5, label='Test Data', zorder=5)

# Linear regression line
y_plot_linear = linear_model.predict(X_plot)
plt.plot(X_plot, y_plot_linear, color=colors[1], linewidth=2,
         label=f'Degree 1 (R²={test_r2_linear:.3f})', linestyle='--')

# Polynomial curves
for degree in degrees:
    y_plot_poly = poly_models[degree].predict(X_plot)
    r2_val = results_df[results_df['Degree'] == degree]['Test R²'].values[0]
    plt.plot(X_plot, y_plot_poly, color=colors[degree], linewidth=2,
             label=f'Degree {degree} (R²={r2_val:.3f})')

plt.xlabel('Engine Displacement (cubic inches)', fontsize=13)
plt.ylabel('Miles Per Gallon (MPG)', fontsize=13)
plt.title('Linear vs Polynomial Regression — Auto MPG Dataset', fontsize=14)
plt.legend(fontsize=11, loc='upper right')
plt.ylim(0, 50)
plt.tight_layout()
plt.show()

---
## Cell 10 — Individual Degree-wise Comparison Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, degree in enumerate(degrees):
    ax = axes[i]
    
    # Data points
    ax.scatter(X_test, y_test, alpha=0.4, s=20, color='gray',
               edgecolors='white', linewidth=0.5, label='Test Data')
    
    # Linear baseline (dashed)
    ax.plot(X_plot, y_plot_linear, color='#E74C3C', linewidth=1.5,
            linestyle='--', alpha=0.6, label=f'Linear (R²={test_r2_linear:.3f})')
    
    # Polynomial curve
    y_plot_poly = poly_models[degree].predict(X_plot)
    r2_val = results_df[results_df['Degree'] == degree]['Test R²'].values[0]
    ax.plot(X_plot, y_plot_poly, color=colors[degree], linewidth=2.5,
            label=f'Degree {degree} (R²={r2_val:.3f})')
    
    ax.set_xlabel('Displacement')
    ax.set_ylabel('MPG')
    ax.set_title(f'Polynomial Degree {degree} vs Linear', fontsize=12)
    ax.legend(fontsize=9)
    ax.set_ylim(0, 50)

plt.suptitle('Degree-wise Comparison: Each Polynomial vs Linear Baseline', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## Cell 11 — Metrics Bar Chart Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

degree_labels = [f'Deg {int(d)}' for d in results_df['Degree']]
bar_colors = [colors[int(d)] for d in results_df['Degree']]
x_pos = np.arange(len(degree_labels))

# ---- MSE Comparison ----
bars1 = axes[0].bar(x_pos - 0.15, results_df['Train MSE'], 0.3,
                     color=bar_colors, alpha=0.5, label='Train')
bars2 = axes[0].bar(x_pos + 0.15, results_df['Test MSE'], 0.3,
                     color=bar_colors, alpha=1.0, label='Test')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(degree_labels)
axes[0].set_ylabel('MSE')
axes[0].set_title('Mean Squared Error')
axes[0].legend()

# ---- RMSE Comparison ----
axes[1].bar(x_pos - 0.15, results_df['Train RMSE'], 0.3,
            color=bar_colors, alpha=0.5, label='Train')
axes[1].bar(x_pos + 0.15, results_df['Test RMSE'], 0.3,
            color=bar_colors, alpha=1.0, label='Test')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(degree_labels)
axes[1].set_ylabel('RMSE')
axes[1].set_title('Root Mean Squared Error')
axes[1].legend()

# ---- R² Comparison ----
axes[2].bar(x_pos - 0.15, results_df['Train R²'], 0.3,
            color=bar_colors, alpha=0.5, label='Train')
axes[2].bar(x_pos + 0.15, results_df['Test R²'], 0.3,
            color=bar_colors, alpha=1.0, label='Test')
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(degree_labels)
axes[2].set_ylabel('R² Score')
axes[2].set_title('R² Score (higher is better)')
axes[2].legend()

plt.suptitle('Metrics Comparison Across All Degrees', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Cell 12 — Residual Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

all_degrees = [1] + degrees  # Include linear
all_preds = [y_test_pred_linear] + [poly_models[d].predict(X_test) for d in degrees]

for i, (degree, y_pred) in enumerate(zip(all_degrees, all_preds)):
    if i >= 5:
        break
    ax = axes.flatten()[i]
    residuals = y_test - y_pred
    
    ax.scatter(y_pred, residuals, alpha=0.4, s=15, color=colors[degree], edgecolors='white', linewidth=0.3)
    ax.axhline(y=0, color='black', linewidth=1, linestyle='--')
    ax.set_xlabel('Predicted MPG')
    ax.set_ylabel('Residual')
    ax.set_title(f'Degree {degree} Residuals')

# Hide the 6th subplot
axes.flatten()[5].axis('off')

plt.suptitle('Residual Plots — Residuals vs Predicted Values', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("Observations:")
print("  Degree 1: Clear curve in residuals → linear model misses the pattern.")
print("  Degree 2: Residuals are more randomly scattered → better fit.")
print("  Degree 3+: Marginal improvement; watch for overfitting at higher degrees.")
print("\n  Human Analogy: Residuals are like 'mistakes after studying.'")
print("  Random mistakes = you understood the concept (good model).")
print("  Patterned mistakes = you missed something systematic (underfitting).")

---
## Cell 13 — Overfitting Detection: Train vs Test R²

In [ ]:
plt.figure(figsize=(10, 6))

all_degrees_list = results_df['Degree'].astype(int).values

plt.plot(all_degrees_list, results_df['Train R²'], 'o-', color='#2ECC71',
         linewidth=2.5, markersize=10, label='Train R²')
plt.plot(all_degrees_list, results_df['Test R²'], 's-', color='#E74C3C',
         linewidth=2.5, markersize=10, label='Test R²')

# Highlight the gap (overfitting indicator)
plt.fill_between(all_degrees_list, results_df['Train R²'], results_df['Test R²'],
                 alpha=0.15, color='red', label='Overfitting Gap')

plt.xlabel('Polynomial Degree', fontsize=13)
plt.ylabel('R² Score', fontsize=13)
plt.title('Overfitting Detection: Train vs Test R² Across Degrees', fontsize=14)
plt.xticks(all_degrees_list)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("How to read this plot:")
print("  • Both lines rising together = model is learning genuinely.")
print("  • Train rising but Test falling = OVERFITTING (memorizing, not learning).")
print("  • The red shaded gap = the 'overfitting zone.'")
print("  • Pick the degree where Test R² is highest and the gap is small.")

---
## Cell 14 — Prediction Demo

In [ ]:
# Predict MPG for sample displacement values
sample_displacements = np.array([100, 150, 200, 250, 300, 350, 400]).reshape(-1, 1)

best_degree = int(results_df.loc[results_df['Test R²'].idxmax(), 'Degree'])

print("=" * 65)
print(f"PREDICTION DEMO (Using Best Model: Degree {best_degree})")
print("=" * 65)
print(f"\n{'Displacement':>15} {'Linear (Deg 1)':>16} {f'Poly (Deg {best_degree})':>16}")
print("-" * 50)

for disp in sample_displacements:
    pred_linear = linear_model.predict(disp.reshape(1, -1))[0]
    if best_degree == 1:
        pred_poly = pred_linear
    else:
        pred_poly = poly_models[best_degree].predict(disp.reshape(1, -1))[0]
    
    print(f"{disp[0]:>12.0f} ci {pred_linear:>13.1f} mpg {pred_poly:>13.1f} mpg")

print("\nNotice how polynomial predictions curve more realistically —")
print("small engines get much better MPG, while large engines plateau.")

---
## Cell 15 — Summary & Key Takeaways

In [ ]:
best_row = results_df.loc[results_df['Test R²'].idxmax()]

print(f"""
╔══════════════════════════════════════════════════════════════════════╗
║                    SUMMARY & KEY TAKEAWAYS                          ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  Dataset:  Auto MPG (392 samples after cleaning)                     ║
║  Feature:  Engine Displacement (cubic inches)                        ║
║  Target:   Miles Per Gallon (MPG)                                    ║
║                                                                      ║
║  Best Model: Degree {int(best_row['Degree'])}                                              ║
║    Test R²:  {best_row['Test R²']:.4f}                                                ║
║    Test MSE: {best_row['Test MSE']:.4f}                                               ║
║                                                                      ║
║  Key Findings:                                                       ║
║  1. Linear regression underfits curved data                          ║
║  2. Degree 2 captures the curve significantly better                 ║
║  3. Higher degrees give diminishing returns and risk overfitting     ║
║  4. The gap between Train R² and Test R² reveals overfitting         ║
║                                                                      ║
║  Human Learning Parallel:                                            ║
║  • Linear = oversimplifying ("all big cars are bad on fuel")         ║
║  • Polynomial = understanding nuance ("it depends on the curve")    ║
║  • High degree = overthinking (memorizing exceptions, not rules)    ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
""")